# Household Power Consumption Forecasting

This notebook builds a time-series regression pipeline to predict hourly `Global_active_power`.

Workflow:
1. Load and clean minute-level readings.
2. Aggregate to hourly values and engineer time/lag features.
3. Train and evaluate a Random Forest baseline.
4. Visualize predictions and feature importance.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "Data").exists():
    ROOT = ROOT.parent

# Load the raw minute-level household dataset
df = pd.read_csv(
    ROOT / "Data" / "household_power_consumption.txt",
    sep=';',
    na_values=['?'],
    low_memory=False
)

# Quick sanity check of size, sample rows, and inferred dtypes
print(df.shape)
print(df.head())
print(df.dtypes)


In [ ]:
# Combine Date and Time into a single datetime column
df['datetime'] = pd.to_datetime(df['Date'] + ' ' + df['Time'], format='%d/%m/%Y %H:%M:%S')

# Set it as the index
df = df.set_index('datetime')

# Drop the now-redundant columns
df = df.drop(columns=['Date', 'Time'])

# Check missing values
print(df.isnull().sum())
print(f"\nTotal rows: {len(df)}")

In [ ]:
# Drop missing rows
df = df.dropna()

# Confirm
print(f"Rows after dropping nulls: {len(df)}")
print(f"Rows removed: {2075259 - len(df)}")

# Quick sanity check - look at our target column
print(f"\nGlobal Active Power stats:")
print(df['Global_active_power'].describe())

In [ ]:
# Resample from per-minute to per-hour (mean)
df_hourly = df['Global_active_power'].resample('h').mean()
df_hourly = df_hourly.to_frame()

print(df_hourly.shape)
print(df_hourly.head())

In [ ]:
# Calendar features encode known usage cycles (hourly routine, weekday/weekend behavior, and seasonality).
df_hourly['hour'] = df_hourly.index.hour
df_hourly['day_of_week'] = df_hourly.index.dayofweek
df_hourly['month'] = df_hourly.index.month
df_hourly['is_weekend'] = df_hourly['day_of_week'].isin([5, 6]).astype(int)

# Lag design rationale:
# - 1h captures short-term persistence (recent consumption is highly predictive).
# - 24h captures daily periodicity (same hour yesterday).
# - 168h captures weekly periodicity (same hour last week).
# These lags are simple, interpretable, and strong baselines before adding advanced features.
df_hourly['lag_1h'] = df_hourly['Global_active_power'].shift(1)
df_hourly['lag_24h'] = df_hourly['Global_active_power'].shift(24)
df_hourly['lag_168h'] = df_hourly['Global_active_power'].shift(168)

# Initial rows without full lag history are removed to prevent target leakage.
df_hourly = df_hourly.dropna()

print(df_hourly.shape)
print(df_hourly.head())

In [ ]:
# Define features and target
X = df_hourly[['hour', 'day_of_week', 'month', 'is_weekend', 'lag_1h', 'lag_24h', 'lag_168h']]
y = df_hourly['Global_active_power']

# Use a chronological split (first 80% train, last 20% test).
# Rationale: random split leaks future patterns into training and overestimates real forecasting performance.
split_index = int(len(df_hourly) * 0.8)

X_train = X.iloc[:split_index]
X_test = X.iloc[split_index:]
y_train = y.iloc[:split_index]
y_test = y.iloc[split_index:]

print(f"Training rows: {len(X_train)}")
print(f"Testing rows: {len(X_test)}")
print(f"Training ends: {X_train.index[-1]}")
print(f"Testing starts: {X_test.index[0]}")

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

# Model choice: Random Forest over linear regression.
# Rationale: energy demand has non-linear interactions (hour x weekday x lag context)
# and tree ensembles capture these without manual interaction terms or strict linearity assumptions.
model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
model.fit(X_train, y_train)

# Make predictions
y_pred = model.predict(X_test)

# Evaluate
mae = mean_absolute_error(y_test, y_pred)
print(f"Mean Absolute Error: {mae:.3f} kW")

# Baseline comparison - what if we just predicted the mean?
baseline_mae = mean_absolute_error(y_test, [y_train.mean()] * len(y_test))
print(f"Baseline MAE (always predict mean): {baseline_mae:.3f} kW")
print(f"Improvement over baseline: {((baseline_mae - mae) / baseline_mae * 100):.1f}%")

In [ ]:
# Plot model predictions against ground truth for the first week of test data
plt.figure(figsize=(15, 5))
plt.plot(y_test.index[:168], y_test.values[:168], label='Actual', color='blue', alpha=0.7)
plt.plot(y_test.index[:168], y_pred[:168], label='Predicted', color='red', alpha=0.7)
plt.title('Energy Consumption: Actual vs Predicted (First 7 Days of Test Set)')
plt.xlabel('Time')
plt.ylabel('Global Active Power (kW)')
plt.legend()
plt.tight_layout()
plt.savefig('predictions_household.png', bbox_inches='tight', dpi=150)
plt.show()


In [ ]:
# Feature importance
features = ['hour', 'day_of_week', 'month', 'is_weekend', 'lag_1h', 'lag_24h', 'lag_168h']
importances = model.feature_importances_

plt.figure(figsize=(10, 5))
plt.barh(features, importances, color='steelblue')
plt.xlabel('Importance Score')
plt.title('What drives energy consumption predictions?')
plt.tight_layout()
plt.savefig('feature_importance_household.png', bbox_inches='tight', dpi=150)
plt.show()

# Print the values too
for feature, importance in sorted(zip(features, importances), key=lambda x: x[1], reverse=True):
    print(f"{feature}: {importance:.3f}")